In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder\
        .appName('DataTransformations')\
        .master('local[*]')\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/23 14:27:36 WARN Utils: Your hostname, Ameys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.190 instead (on interface en0)
26/03/23 14:27:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 14:27:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
data = [
    (1, "Alice", "Laptop", 1200, "US"),
    (2, "Bob", "Phone", 800, "US"),
    (3, "Alice", "Mouse", 25, "UK"),
    (4, "David", "Laptop", 1300, "US"),
    (5, "Eve", "Keyboard", 75, "IN"),
    (6, "Frank", "Phone", 900, "US"),
    (7, "Alice", "Laptop", 1150, "UK")
]

columns = ["user_id", "user_name", "product", "amount", "country"]


orders_df = spark.createDataFrame(data,columns)

In [10]:
from pyspark.sql.functions import col, when, lower



In [8]:
orders_df.filter(col('country')=='US')\
         .select("user_name","product","amount")\
         .withColumn('high_value',
                    when(col('amount')>1000,'yes').otherwise('No')
                    ).explain()

== Physical Plan ==
*(1) Project [user_name#1, product#2, amount#3L, CASE WHEN (amount#3L > 1000) THEN yes ELSE No END AS high_value#20]
+- *(1) Filter (isnotnull(country#4) AND (country#4 = US))
   +- *(1) Scan ExistingRDD[user_id#0L,user_name#1,product#2,amount#3L,country#4]




In [12]:
data = [
    (1, "Alice", "Laptop", 1200, "US", "completed"),
    (2, "Bob", "Phone", 800, "US", "completed"),
    (3, "Alice", "Mouse", 25, "UK", "returned"),
    (4, "David", "Laptop", 1300, "US", "completed"),
    (5, "Eve", "Keyboard", 75, "IN", "pending"),
    (6, "Frank", "Phone", 900, "US", "completed"),
    (7, "Alice", "Laptop", 1150, "UK", "completed"),
    (8, "Bob", "Monitor", 300, "US", "pending")
]

columns = ["user_id", "user_name", "product", "amount", "country", "status"]
orders_df_new = spark.createDataFrame(data,columns)

In [17]:
orders_df_new.withColumn('new_amt',when(col('amount')>=1000,'distcount')).show()

+-------+---------+--------+------+-------+---------+---------+
|user_id|user_name| product|amount|country|   status|  new_amt|
+-------+---------+--------+------+-------+---------+---------+
|      1|    Alice|  Laptop|  1200|     US|completed|distcount|
|      2|      Bob|   Phone|   800|     US|completed|     NULL|
|      3|    Alice|   Mouse|    25|     UK| returned|     NULL|
|      4|    David|  Laptop|  1300|     US|completed|distcount|
|      5|      Eve|Keyboard|    75|     IN|  pending|     NULL|
|      6|    Frank|   Phone|   900|     US|completed|     NULL|
|      7|    Alice|  Laptop|  1150|     UK|completed|distcount|
|      8|      Bob| Monitor|   300|     US|  pending|     NULL|
+-------+---------+--------+------+-------+---------+---------+



In [21]:
orders_df_new \
    .select("user_name", "product", "status", "amount", "country") \
    .filter(col("status") == "completed") \
    .withColumn(
        "discount_flag",
        when(col("amount") >= 1000, "discount").otherwise("regular")
    ) \
    .withColumn("country_code", lower(col("country"))) \
    .select("user_name", "product", "amount", "country_code", "discount_flag") \
    .show()

+---------+-------+------+------------+-------------+
|user_name|product|amount|country_code|discount_flag|
+---------+-------+------+------------+-------------+
|    Alice| Laptop|  1200|          us|     discount|
|      Bob|  Phone|   800|          us|      regular|
|    David| Laptop|  1300|          us|     discount|
|    Frank|  Phone|   900|          us|      regular|
|    Alice| Laptop|  1150|          uk|     discount|
+---------+-------+------+------------+-------------+



In [22]:
from pyspark.sql.functions import col, when, lower, upper, lit

data = [
    (1, "Alice", "Laptop", 1200, "US", "completed"),
    (2, "Bob", "Phone", 800, "US", "completed"),
    (3, "Alice", "Mouse", 25, "UK", "returned"),
    (4, "David", "Laptop", 1300, "US", "completed"),
    (5, "Eve", "Keyboard", 75, "IN", "pending"),
    (6, "Frank", "Phone", 900, "US", "completed"),
    (7, "Alice", "Laptop", 1150, "UK", "completed"),
    (8, "Bob", "Monitor", 300, "US", "pending"),
    (9, "George", "Tablet", 600, "CA", "completed")
]

columns = ["user_id", "user_name", "product", "amount", "country", "status"]

orders_df = spark.createDataFrame(data, columns)

In [23]:
orders_df.show()

+-------+---------+--------+------+-------+---------+
|user_id|user_name| product|amount|country|   status|
+-------+---------+--------+------+-------+---------+
|      1|    Alice|  Laptop|  1200|     US|completed|
|      2|      Bob|   Phone|   800|     US|completed|
|      3|    Alice|   Mouse|    25|     UK| returned|
|      4|    David|  Laptop|  1300|     US|completed|
|      5|      Eve|Keyboard|    75|     IN|  pending|
|      6|    Frank|   Phone|   900|     US|completed|
|      7|    Alice|  Laptop|  1150|     UK|completed|
|      8|      Bob| Monitor|   300|     US|  pending|
|      9|   George|  Tablet|   600|     CA|completed|
+-------+---------+--------+------+-------+---------+



In [25]:
orders_df.filter(col('status')!='returned')\
         .withColumn('user_name_upper', upper(col('user_name')))\
         .withColumn('user_band',
             when(col('amount')>=1000,"high")\
            .when((col('amount')>=500) & (col('amount')<1000), "medium")\
            .otherwise("low")
           )\
         .withColumn('is_us_order',
                     when(col('country')=='US',"yes").otherwise('no')
                    )\
         .select('user_id','user_name_upper','product','amount','user_band','is_us_order')\
         .show()

+-------+---------------+--------+------+---------+-----------+
|user_id|user_name_upper| product|amount|user_band|is_us_order|
+-------+---------------+--------+------+---------+-----------+
|      1|          ALICE|  Laptop|  1200|     high|        yes|
|      2|            BOB|   Phone|   800|   medium|        yes|
|      4|          DAVID|  Laptop|  1300|     high|        yes|
|      5|            EVE|Keyboard|    75|      low|         no|
|      6|          FRANK|   Phone|   900|   medium|        yes|
|      7|          ALICE|  Laptop|  1150|     high|         no|
|      8|            BOB| Monitor|   300|      low|        yes|
|      9|         GEORGE|  Tablet|   600|   medium|         no|
+-------+---------------+--------+------+---------+-----------+

